<a href="https://colab.research.google.com/github/Helena-Abella/Shell-and-Tube-Heat-Exchanger-Design/blob/main/Multi_Objective_Shell_and_Tube_Heat_Exchanger_Optimization_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import math as m
import matplotlib.pyplot as plt
from tabulate import tabulate
import os


def upload_doc(file_name, sheet=None):
  if sheet:
      df = pd.read_excel(file_name, sheet_name=sheet, header=None)
  else:
      df = pd.read_excel(file_name, header=None)
  return df.values.tolist()

def upload_table():
    archives = {
        'C1': ('Appendix C.xlsx', 'C1'),
        'C2': ('Appendix C.xlsx', 'C2'),
        'C3': ('Appendix C.xlsx', 'C3'),
        'C4': ('Appendix C.xlsx', 'C4'),
        'C5': ('Appendix C.xlsx', 'C5'),
        'C6': ('Appendix C.xlsx', 'C6'),
        'C7': ('Appendix C.xlsx', 'C7'),
        'C8': ('Appendix C.xlsx', 'C8'),
        'B1': ('Appendix B.xlsx', 'Table 1')}

    tables = {}

    for name, (archive, sheet) in archives.items():
        data = upload_doc(archive, sheet)
        tables[name] = data

    return tables

In [ ]:
#Correlation for shell-side heat-transfer coefficient.

Correlation = [[0.75,1,'Square',0.250,0.95],[1,1.25,'Square',0.250,0.99],[1.25,1.5625,'Square',0.3125,1.23],[1.5,1.875,'Square',0.375,1.48],
              [0.625,0.8125,'Square',0.1875,0.535],[0.75,0.9375,'Triangular',0.1875,0.55],[0.75,1,'Triangular',0.250,0.73],[1,1.25,'Triangular',0.250,0.72],
              [1.25,1.5625,'Triangular',0.3125,0.91],[1.5,1.875,'Triangular',0.375,1.08]]

headers = ['Tube OD  (in)','Pitch (in)','layout', 'Clearance (in)', 'Deq (in)']

In [ ]:
#Guidelines for Sizing Nozzles

n_data = [((4,10), 2),((12, 17.25), 3),((19.25, 21.25), 4),((23, 29), 6),((30, 37), 8),((39, float('inf')), 10)]
headers = ['Shell size (in)','Nominal nozzle diameter (in)']

def get_Dn(Ds):
    for range, Dn in n_data:
      min,max = range
      if min <= Ds <= max:
        return Dn, n_data.index((range,Dn))

In [ ]:
# Tabla 5.2 Standard Values (Inches) for Baffle Thickness in Class R Heat Exchangers

Bt_data = {(8, 14): [((0, 24), 0.125),((24, 36), 0.1875),((36, 48), 0.250),((48, 60), 0.375),((60, float('inf')), 0.375)],
          (15, 28): [((0, 24), 0.1875),((24, 36), 0.250),((36, 48), 0.375),((48, 60), 0.375),((60, float('inf')), 0.500)],
          (29, 38): [((0, 24), 0.250),((24, 36), 0.3125),((36, 48), 0.375),((48, 60), 0.500),((60, float('inf')), 0.625)],
          (39, 60): [((0, 24), 0.250),((24, 36), 0.375),((36, 48), 0.500),((48, 60), 0.625),((60, float('inf')), 0.625)],
          (61, 100): [((0, 24), 0.375),((24, 36), 0.500),((36, 48), 0.625),((48, 60), 0.750),((60, float('inf')), 0.750)],}

def get_Bt(Ds, B):
    for (Ds_min, Ds_max), ranks in Bt_data.items():
        if Ds_min <= Ds <= Ds_max:
            for ((B_min, B_max), Bt) in ranks:
                if B_min <= B <= B_max:
                    return Bt

In [ ]:
def getj(layout,P_T,DE_t,Nc,vB):

  #Gnielinski's correlations

  if layout == 'Square':                                                        #layout Square 90°
    s1 = s2 = P_T
  else:
    s2 = P_T                                                                    #Transverse (in)
    s1 = P_T*m.cos(m.radians(30))                                               #Longitudinal (in)

  a = s1/DE_t
  b = s2/DE_t

  if b >= 1:
    Ψ = 1-m.pi/4/a                                                              #Void fraction
  else:
    Ψ = 1-m.pi/4/a/b

  l = m.pi*DE_t/2/12
  Re = vB*3600*l/Ψ/mu_s*rho_s
  Pr = mu_s/rho_s/a

  Nu_lam = 0.664*m.sqrt(Re)*(Pr**(1/3))                                         #Laminar flux Nusselt
  Nu_turb = 0.037*Re**0.8*Pr/(1 + 2.443*Re**(-0.1)*(Pr**(2/3) - 1))             #Turbulent flux Nusselt

  if layout == 'Square':                                                        #In-line arrangement
    if b >= 1.2:
        f_A = 1 + 0.7*(b/a - 0.3)/(Ψ**1.5)/((b/a + 0.7)**2)                     #Arrangement factor
  else:                                                                         #Triangular (staggered)
      f_A = 1 + 2/3/b


  Nu_l0 = 0.3 + m.sqrt(Nu_lam**2 + Nu_turb**2)
  if Nc >= 10:
    Nu_b  = f_A*Nu_l0
  else:
    Nu_b = (1 + (Nc-1)*f_A)/Nc*Nu_l0

  j = Nu_b/Re/(Pr**(1/3))                                                       #Colburn factor

  return j

In [ ]:
def iterate(m_B,D0,omega,C,SBW,PT,b,Ds,Bc,m_s,omega2,Nss,Sbp,Sw,Sm,Bt,dtb,St,dsb,Ss,gc,a,Dotl):

  error = 1

  while error > 0.001:
    Dv = (omega*PT**2-D0**2)/D0   #in
    ξB = 4*a*D0*Ds*Dv*(1-2*Bc)*(C)**(-3)*(m_B*D0/12/mu_s/SBW)**(-b)/2/rho_s/gc/(SBW**2)             #Cross-flow resistance (lbf*s^2/lb^2ft^2)

    ξCF = (0.266*Ds*((1-2*Bc)/omega2/PT)+2*Nss)/2/rho_s/gc/(Sbp**2)                                 #(lbf*s^2/lb^2ft^2)

    ξw = 1.9*np.exp(0.6856*Sw/Sm)/2/rho_s/gc/(Sw**2)                                                #Window flow resistance  (lbf*s^2/lb^2ft^2)

    ξx =(ξB**(-1/2)+ξCF**(-1/2))**(-2)                                                              #(lbf*s^2/lb^2ft^2)

    ξy = ξw + ξx                                                                                    #(lbf*s^2/lb^2ft^2)

    ξA = (0.036*Bt/dtb+2.3*(Bt/dtb)**(-0.177))/2/rho_s/gc/(St**2)                                   #Tube-to-baffle leakage flow resistance (lbf*s^2/lb^2ft^2)

    ξE = (0.036*Bt/dsb+2.3*(Bt/dsb)**(-0.177))/2/rho_s/gc/(Ss**2)                                   #Shell-to-baffle leakage flow resistance (lbf*s^2/lb^2ft^2)

    ξ_0 = (ξA**(-1/2)+ξE**(-1/2)+ξy**(-1/2))**(-2)                                                  #(lbf*s^2/lb^2ft^2)

    ξe = 0.5*ξx*(1**2)*(1+Dotl/(Ds*(1-2*Bc)))                                                       #Cross-flow resistance in end (inlet or outlet) baffle space (lbf*s^2/lb^2ft^2)
                                                                                                    #B/Be=1 with Be as end (inlet or outlet) baffle spacing
    ξwe = 1.9*np.exp(0.6856*Sw/Sm*1)/2/rho_s/gc/(Sw**2)                                             #Flow resistance in end (inlet or outlet) baffle window (lbf*s^2/lb^2ft^2)


    m_B2 = m_s*m.sqrt(ξx*ξ_0/ξB/ξy)     #lb/h

    m_w = m_s*m.sqrt(ξ_0/ξy)            #lb/h
    m_A = m_s*m.sqrt(ξ_0/ξA)            #lb/h
    m_CF = m_s*m.sqrt(ξx*ξ_0/ξCF/ξy)    #lb/h
    m_E = m_s*m.sqrt(ξ_0/ξE)            #lb/h
    m_e = 0.5*(m_s+m_w)                 #lb/h

    error = abs(m_B2 - m_B) / m_B
    m_B = m_B2

  return m_B,m_w,m_A,m_CF,m_E,m_e,Dv,ξB,ξCF,ξA,ξE,ξw,ξx,ξy,ξ_0,ξe,ξwe

In [ ]:
#Design data
#Crude (Shell-side)
M_s = 873030.55825                    #Shell-side mass flux (lb/h)
T_1 = 194                             #°F
T_2 = 122                             #°F
R_s = 1.13565*10**(-3)                #hft^2°F/BTU
dPmáx_s = 10.15                       #TEMA - Liquid: 1.45-10.15 psi  Gas: < 7.25 psi
dPmín_s = 1.45                        #psi
v_smín = 0.98                         #TEMA - Liquid: 0.98-3.28 ft/s  Gas: 9.84-49.21 ft/s
v_smax = 3.28                         #ft/s

#Water (Tube-side)
M_t = 1815903.5612                    #Tube-side mass flux (lb/h)
t_1 = 86                              #°F
t_2 = 104                             #°F
R_t = 2.2713*10**(-3)                 #hft^2°F/BTU
dPmáx_t = 14.50                       #TEMA - Liquid: 2.90-14.5psi  Gas: Keep as low as possible
dPmín_t = 2.90                        #psi
v_tmín = 3.28                         #ft/s  # TEMA - Liquid: 3.28-6.56 ft/s  Gas: 32.81-98.42 ft/s
v_tmax = 6.56                         #ft/s

In [ ]:
#Physical properties of the streams
#Crude
rho_s = 49.093348197                  #Density lb/ft^3
Cp_s = 0.51829559568                  #Specific heat capacity BTU/lb°F
mu_s = 4.5720766106                   #Dynamic viscosity lb/fth
k_s = 0.07053746986                   #Thermal conductivity BTU/hft°F
Pr_s = Cp_s*mu_s/k_s                  #Prandtl number
s_s = rho_s/62.427960576              #Specific gravity

#Agua
rho_t = 62.115820773                  #Density lb/ft^3
Cp_t = 1.0000477692                   #Specific heat capacity BTU/lb°F
mu_t = 1.7417434707                   #Dynamic viscosity lb/fth
k_t = 0.34112382965                   #Thermal conductivity BTU/hft°F
Pr_t = Cp_t*mu_t/k_t                  #Prandtl number
s_t = rho_t/62.427960576              #Specific gravity
phi = 1                               #Viscosity correction factor
k_wall = 26                           #Acero al carbono

In [ ]:
#Condiciones del sistema

dT_1 = T_1-t_2
dT_2 = T_2-t_1
MLDT = (dT_2-dT_1)/m.log(dT_2/dT_1)
R = (T_1-T_2)/(t_2-t_1)                   #Temperature ratio
P = (t_2-t_1)/(T_1-t_1)                   #Thermal effectiveness



Q = M_s*Cp_s*(T_1 - T_2)                  #BTU/h Total heat duty

In [ ]:
def properties(tabla, DE_t, layout, P_T, N, F, B1_data):
  import pandas as pd
  global overall_results, Bc_ind
  B1 = B1_data
  row = 0                                   #Table C
  column = 0                                #Table C
  i = 6                                     #Minimum industrial tube length (ft)
  f = 0                                     #Table B
  c = 0
  Bc_ind = [0.20,0.25,0.30,0.35]


  for row in range(2,len(tabla)):
    for column in range(1,len(tabla[0])):
      for B_Ds in np.arange(0.20,1,0.05):
        for f in range(1,len(B1)):
          line = []
          if float(B1[f][0]) == DE_t:
            specifications(6,line,DE_t,layout,P_T,tabla,column,row,B1,f,B_Ds,overall_results,N,F)
            for i in range(8,25,4):
              specifications(i,[],DE_t,layout,P_T,tabla,column,row,B1,f,B_Ds,overall_results,N,F)

In [ ]:
def specifications(L_t,line,DE_t,layout,P_T,tabla,column,row,B1,f,B_Ds,resultados,N,F):

  line.append(N)
  line.append(DE_t)                                                             #in
  line.append(layout)
  line.append(P_T)                                                              #in
  headtype = tabla[0][column]                                                   #Head type
  line.append(headtype)
  line.append(L_t)                                                              #ft
  n_p = float(tabla[1][column])                                                 #Number of passes
  line.append(n_p)
  DI_s = float(tabla[row][0])                                                   #Shell inside diameter (in)
  line.append(DI_s) #in
  if headtype == 'U':
    n_t = float(tabla[row][column])/2                                           #Number of tubes
  else:
    n_t  = float(tabla[row][column])                                            #Number of tubes
  line.append(n_t)
  eps = float(B1[f][2])                                                         #Tube thickness (in)
  line.append(eps)                                                              #in
  DI_t = float(B1[f][3])/12                                                     #Tube inside diameter (ft)

  ld = L_t*12/DI_s
  if ld < 5 or ld > 10:
    return

  #Baffles
  B = float(B_Ds*DI_s)                                                          #Baffle spacing (in)
  if B < DI_s/5 or B > DI_s or  B < 2:
    return
  line.append(B) #in
  Bc_calc = -0.6735639*B_Ds**2 + 21.94439185*B_Ds + 15.05885461

  Bc = min(Bc_ind, key=lambda x: abs(x - Bc_calc))

  n_b = m.ceil(L_t*12/B-1)                                                      #Number of baffles
  line.append(n_b)


  equations(n_t,n_p,DI_t,line,L_t,P_T,DE_t,DI_s,B,layout,headtype,n_b,N,F,Bc,eps)

  for i in line:
    if i == 0:
      return

  overall_results.append(line)

In [ ]:
def equations(n_t,n_p,DI_t,line,L_t,P_T,DE_t,DI_s,B,layout,headtype,n_b,N,F,Bc,eps):

#Sieder-Tate and Hausen equations
#Tube-side pressure drop

  if n_t == 0:
    return

  m_s = M_s/N                                                                   #Shell-side mass flux per unit (lb/h)
  m_t = M_t/N                                                                   #Tube-side mass flux per unit (lb/h)
  q = Q/N

  G_t = 4*m_t*n_p/n_t/m.pi/(DI_t**2)                                            #Tube mass velocity (lb/hft^2)
  v_t = G_t/rho_t #ft/h
  line.append(v_t/3600)


  if v_t/3600 < v_tmín or v_t/3600 > v_tmax:
    line.append(0)
    return


  Re_t = G_t*DI_t/mu_t


  if Re_t >= 3000:
    f_t = 0.4137*(Re_t**(-0.2585))
    if headtype == 'U':
      alpha_r = 1.6*n_p-1.5
    else:
      alpha_r = 2*n_p-1.5
  elif Re_t < 2100:
    f_t = 64/Re_t
    if headtype == 'U':
      alpha_r = 2.38*n_p-1.5
    else:
      alpha_r = 3.25*n_p-1.5
  else:
    line.append(0)
    return


  dP_ft = f_t*n_p*L_t*(G_t**2)/(7.5*10**12)/DI_t/s_t/phi
  dP_r = (1.334*10**(-13))*alpha_r*(G_t**2)/s_t
  dP_t = dP_r + dP_ft
  line.append(dP_t)

  if dP_t < dPmín_t or dP_t > dPmáx_t:
    line.append(0)
    return


  #Shell-side pressure drop

  for row in Correlation:
    if row[0] == DE_t and row[1] == P_T and row[2] == layout:
      Deq = row[4]   #in
      break

  for row in Correlation:
    if row[0] == DE_t and row[1] == P_T and row[2] == layout:
      C = row[3]
      break

  if headtype == 'P o S':
    Dotl = DI_s/0.0394 - (25.303307 + 0.01643*(DI_s/0.0394))                    #Outer tube limit diameter  (mm)
  else:
    Dotl = DI_s/0.0394 - (12.807573 + 0.004904*(DI_s/0.0394))

  Dotl = Dotl*0.0394
  D_ctl = Dotl - DE_t                                                           #Central tube limit diameter   (in)

  if abs(DI_s*(1-2*Bc)/D_ctl) > 1:
    line.append(0)
    return
  theta_ctl = 2*m.acos(DI_s*(1-2*Bc)/D_ctl)                                     #rad
  Fc = 1 + 1/m.pi*(m.sin(theta_ctl)-theta_ctl)                                  #Fraction of tubes in cross flow between baffle tips
  Jc = 0.55 + 0.72*Fc                                                           #Heat-transfer correction factor for effect of baffle window flow


  PT_eff = P_T                                                                  #Triangular and 90° square layouts (in)
  Sm_in = B*(DI_s-Dotl+D_ctl/PT_eff*(C))
  Sm = Sm_in/144                                                                #Cross-flow area (ft^2)

  theta_ds = 2*m.acos(1-2*Bc)
  safety = 0.75                                                                 #(mm) To account for out-of-roundness tolerances in both the shell and the baffles, a safety factor of 0.75 mm is often added
  dsb_mm = 0.8 + 0.002*DI_s*25.4 + safety
  dsb = dsb_mm/25.4                                                             #Shell-to-baffle clearance (in)
  S_sb_in = DI_s*dsb*(m.pi-0.5*theta_ds)
  S_sb = S_sb_in/144                                                            #Shell-to-baffle leakage area (ft^2)

  Ss_in = m.pi*DI_s*dsb
  Ss = Ss_in/144                                                                #Approximate flow area for shell-to-baffle leakage stream (ft^2)

  if DE_t > 1.25:
    dtb_mm = 0.4
  else:
    if 2*B/12 < 3:
      dtb_mm = 0.4
    else:
      dtb_mm = 0.2

  dtb = dtb_mm*0.0394               #in
  S_tb_in = 0.5*m.pi*DE_t*dtb*n_t*(1 + Fc)
  S_tb = S_tb_in/144                                                            #Bundle bypass flow area (ft^2)
  rs = S_sb/(S_sb+S_tb)
  r1 = (S_sb+S_tb)/Sm
  J_L = 0.44*(1-rs) + (1-0.44*(1-rs))*np.exp(-2.2*r1)                           #Heat-transfer correction factor for baffle leakage effects

  if J_L < 0.2 or J_L > 1:
    line.append(0)
    return

  p = 0.8-0.15*(1 + rs)
  R_L = np.exp(-1.33*(1+rs)*r1**p)


  G_s = m_s/Sm                                                                  #Mass flux of shell-side fluid based on cross-flow area (lb/hft^2)
  Re_s = Deq/12*G_s/mu_s
  Sb_in = B*(DI_s-Dotl)
  Sb = Sb_in/144                                                                #Bundle bypass flow area (ft^2)
  theta_otl = 2*m.acos(DI_s*(1-2*Bc)/Dotl)
  Np = 0                                                                        #Number of tube pass partitions aligned in cross-flow direction  (0:tube pass partitions are normal to the cross-flow direction) (>0: paralelo)
  dp = 0                                                                        #Tube-pass-partition clearance (conceptual)

  SBW_in = B*Dotl**2*(m.pi-theta_otl+m.sin(theta_otl))/4/DI_s/(1-2*Bc)-B*Np*dp  #in^2
  SBW = SBW_in/144     #ft^2

  Sbp_in = B*(DI_s-Dotl+Np*dp)
  Sbp = Sbp_in/144                                                              #Flow area for combined bypass stream (ft^2)


  Fw = (1-Fc)/2
  A_t = n_t*Fw*m.pi*DE_t**2/4                                                   #in^2
  Swg = 1/8*DI_s**2*(theta_ds-m.sin(theta_ds))                                  #in^2
  Sw_in = Swg - A_t
  Sw = Sw_in/144                                                                #Window flow area (ft^2)


  St_in = n_t*m.pi*DE_t*dtb
  St = St_in/144                                                                #ft^2

  Bt = get_Bt(DI_s, B)                                                          #in


  gc = 32.174

  if layout == 'Square':                                                        #90° Square layout
    angulo = 90
    a = 0.061
    b = 0.088
    omega = 1.273
    omega2 = 1
    PT_prima = P_T
    Nc = DI_s*(1-2*Bc)/PT_prima                                                 #Number of tube rows crossed in flow between two baffle tips
    Nss = m.ceil(Nc / 10)                                                       #Number of pairs of sealing strips
    m_B,m_w,m_A,m_CF,m_E,m_e,Dv,ξB,ξCF,ξA,ξE,ξw,ξx,ξy,ξ_0,ξe,ξwe = iterate(0.5*m_s,DE_t,omega,C,SBW,P_T,b,DI_s,Bc,m_s,omega2,Nss,Sbp,Sw,Sm,Bt,dtb,St,dsb,Ss,gc,a,Dotl)
    if m_B < 0:
      line.append(0)
      return
  else:
    angulo = 30
    a = 0.450
    b = 0.267
    omega = 1.103
    omega2 = 1.732
    PT_prima = P_T*m.cos(30)                                                    #Tube pitch parallel to flow direction (in)
    Nc = DI_s*(1-2*Bc)/PT_prima
    Nss = m.ceil(Nc / 10)
    m_B,m_w,m_A,m_CF,m_E,m_e,Dv,ξB,ξCF,ξA,ξE,ξw,ξx,ξy,ξ_0,ξe,ξwe = iterate(0.5*m_s,DE_t,omega,C,SBW,P_T,b,DI_s,Bc,m_s,omega2,Nss,Sbp,Sw,Sm,Bt,dtb,St,dsb,Ss,gc,a,Dotl)
    if m_B < 0:
      line.append(0)
      return

  r_ss = Nss/Nc

  if Re_s < 100:
    CJ = 1.35
    CR =4.5
    n1 = 1/3
    n2 = 1
  else:
    CJ = 1.25
    CR = 3.7
    n1 = 0.6
    n2 = 0.2

  if r_ss < 0.5:
    J_B = np.exp(-CJ*Sb/Sm*(1-m.cbrt(2*r_ss)))
    R_B = np.exp(-CR*Sb/Sm*(1-m.cbrt(2*r_ss)))
  else:
    J_B = 1
    R_B = 1

  Bin = (L_t*12-(n_b-1)*B-eps-0.12)/2
  Bout = Bin
  J_S = ((n_b-1)+(Bin/B)**(1-n1)+(Bout/B)**(1-n1))/((n_b-1)+(Bin/B)+(Bout/B))
  R_S = 0.5*((B/Bin)**(2-n2)+(B/Bout)**(2-n2))


  Ncw = 0.8*Bc*DI_s/PT_prima                                                    #Number of tube rows crossed in flow through shell
  Nct = (n_b+1)*(Nc+Ncw)                                                        #Effective number of tube rows crossed in flow through one baffle window
  if Re_s <= 20:
    J_R = (10/Nct)**0.18
  elif Re_s >= 100:
    J_R = 1
  else:
    J_R = (10/Nct)**0.18 + (Re_s-20) * (1.0-(10/Nct)**0.18)/80                  #Interpolation

  if J_R < 0.4 or J_R > 1:
    line.append(0)
    return


  vB = m_B/Sm/rho_s/3600                                                        #ft/s
  line.append(vB)
  Re_B = DE_t/12*m_B/mu_s/Sm

  if vB < v_smín or vB > v_smax:
    line.append(0)
    return

  if Re_B < 100:
    line.append(0)
    return
  elif Re_B >= 1000:
    ψ = 1
  else:
    ψ = 3.646*Re_B**(-0.1934)

  dPy = ξ_0*(m_s/3600)**2                                                       #Pressure drop across one baffle space and one baffle window (lbf/ft^2)
  dPe = ξe*(m_e/3600)**2+0.5*ξwe*(m_w/3600)**2                                  #Pressure drop in end (inlet or outlet) baffle space (lbf/ft^2)

  if m_s/m_t <= 0.5:
    Dn_s = n_data[get_Dn(DI_s)[1]-1][1]
  else:
    Dn_s = get_Dn(DI_s)[0]

  Re_ns = 4*m_s/m.pi/(Dn_s/12)/mu_s
  G_ns = 4*m_s/m.pi/((Dn_s/12)**2)
  if Re_ns >= 4000:
    dPn_s = 2e-13*1*G_ns**2/s_s                                                 #Pressure drop in shell nozzles (psi)(El 1 es por Ns, mi código asume no hay corazas en serie)
  elif 4000 > Re_ns >= 100:
    dPn_s = 4e-13*1*G_ns**2/s_s

  dPs = (ψ*((n_b-1)*dPy + 2*dPe))/144 + dPn_s
  line.append(dPs)

  if dPs < dPmín_s or dPs > dPmáx_s:
    line.append(0)
    return

  #Heat-transfer

  hid = getj(layout,P_T,DE_t,m.ceil(Nc),vB)*Cp_s*G_s*phi*Pr_s**(-2/3)
  h0 = hid*Jc*J_L*J_B*J_S*J_R                                                   #Shell-side heat transfer coefficient (BTU/hft^2°F)
  line.append(h0)
  hi = (k_t/DI_t)*0.023*Re_t**(0.8)*Pr_t**(1/3)*phi                             #Tube-side heat transfer coefficient (BTU/hft^2°F)
  line.append(hi)

  R_d = R_t*DE_t/DI_t/12 + R_s                                                  #Total fouling factor hft^2°F/BTU
  U_d = (DE_t/hi/DI_t/12 + DE_t/12*m.log(DE_t/DI_t/12)/2/k_wall + 1/h0 + R_d)**(-1)                                                       #Design overall heat-transfer coefficient (BTU/hft^2°F) (considera ensuciamiento)
  line.append(U_d)
  U_c = (1/U_d-R_d)**(-1)                                                       #Clean overall heat-transfer coefficient (BTU/hft^2°F)
  line.append(U_c)
  U_req = q*12/n_t/m.pi/DE_t/L_t/F/MLDT

  A_req = q/U_d/F/MLDT
  line.append(A_req)

  A_i = n_t*m.pi*DE_t/12*L_t                                                    #ft^2
  line.append(A_i)

  A = A_i*N                                                                     #ft^2
  line.append(A)

  A_ratio = A_i/A_req
  line.append(A_ratio)

  if A_ratio < 1.10 or A_ratio > 1.20:
    line.append(0)
    return

In [ ]:
overall_results = []
variables = ['Number of heat exchangers','Tube outside diameter (in)','Tube layout','Tube pitch (in)','Tube head type','Tube length (ft)',
             'Number of tube passes','Shell inside diameter (in)','Number of tubes','Thickness (in)','Baffle spacing (in)','Number of baffles',
             'Tube-side velocity (ft/h)','Tube-side pressure drop (psi)','Shell-side velocity (ft/h)','Shell-side pressure drop (psi)',
             'Shell-side heat transfer coefficient (BTU/h·ft²·°F)','Tube-side heat transfer coefficient (BTU/h·ft²·°F)',
             'Design overall heat transfer coefficient (BTU/h·ft²·°F)','Clean overall heat transfer coefficient (BTU/h·ft²·°F)',
             'Required area (ft²)','Unit area (ft²)','Area (ft^2)','Area ratio']


tables = upload_table()
B1_data = tables.get('B1',[])

for N in range(1,6):   #Number of heat exchangers
    if R == 1:
      S = P
      F = S*m.sqrt(2)/(1-S)/m.log((2-S*(2-m.sqrt(2)))/(2-S*(2+m.sqrt(2))))
    else:
      a = (1-R*P)/(1-P)
      S = (a-1)/(a-R)
      F = m.sqrt(R**2+1)*m.log((1-S)/(1-R*S))/(R-1)/m.log((2-S*(R+1-m.sqrt(R**2+1)))/(2-S*(R+1+m.sqrt(R**2+1))))
    if F >= 0.8:
      configs = [('C1', tables.get('C1', []), 0.625, 'Square', 0.8125), ('C2', tables.get('C2', []), 0.75, 'Triangular', 0.9375),
                 ('C3', tables.get('C3', []), 0.75, 'Square', 1),('C4', tables.get('C4', []), 0.75, 'Triangular', 1),
                 ('C5', tables.get('C5', []), 1, 'Square', 1.25), ('C6', tables.get('C6', []), 1, 'Triangular', 1.25),
                 ('C7', tables.get('C7', []), 1.25, 'Square', 1.5625), ('C8', tables.get('C8', []), 1.25, 'Triangular', 1.5625)]
      for nombre, tabla, DE_t, layout, P_T in configs:
        properties(tabla, DE_t, layout, P_T,N, F, B1_data)
    else:
      continue

df_master = pd.DataFrame(overall_results, columns=variables)
df_master.to_excel("overall_results.xlsx", index=False)

In [ ]:
pareto = []
total = len(df_master)

df_master['ΔP_total (psi)'] = df_master['Tube-side pressure drop (psi)'] + df_master['Shell-side pressure drop (psi)']

for i, row_i in df_master.iterrows():                                           #Compare configurations (i y j)
    dominated = False
    for j, row_j in df_master.iterrows():                                       #Condition to determine whether j dominates i
        if ((row_j['Area (ft^2)'] <= row_i['Area (ft^2)']) and (row_j["ΔP_total (psi)"] <= row_i["ΔP_total (psi)"]) and
            ((row_j['Area (ft^2)'] < row_i['Area (ft^2)']) or (row_j["ΔP_total (psi)"] < row_i["ΔP_total (psi)"]))):
            dominated = True
            break
    if not dominated:
        pareto.append(row_i)


df_pareto = pd.DataFrame(pareto)
print('Number of Pareto front solutions:', len(df_pareto))

df_pareto.to_excel('Pareto_front.xlsx', index=False)
plt.figure()

plt.scatter(df_master['Area (ft^2)'], df_master['ΔP_total (psi)'], c='#E8E0F0', s=15)
plt.scatter(df_pareto["Area (ft^2)"], df_pareto["ΔP_total (psi)"],c='#4A148C', s=25)

plt.xlabel("Area (ft²)", fontsize=14, fontweight='bold')
plt.ylabel("ΔP (psi)", fontsize=14, fontweight='bold')


plt.grid(True, linestyle='--', alpha=0.5, linewidth=0.8)

plt.show()

In [ ]:
data = df_pareto[['Area (ft^2)','ΔP_total (psi)']].values

#Data normalization
min_val = data.min(axis=0)
max_val = data.max(axis=0)

data_norm = (data - min_val) / (max_val - min_val)

ideal = [0,0]

distances = np.linalg.norm(data_norm - ideal, axis=1)

idx_knee = np.argmin(distances)

knee_point = df_pareto.iloc[idx_knee]

print("Found Knee Point:")
print(knee_point)